# Long-Short Backtesting Class

上一节的 `BacktestLongOnly` 只能在空仓与多头之间切换。书中 "Long-Short Backtesting Class" 给出 `BacktestLongShort`：同样继承 `BacktestBase`，仓位可取 `1`（多）、`0`（中性）、`-1`（空）。

相对纯多头，多空版多了 `go_long` / `go_short`：若已持反向仓，先平仓再开仓。SMA 与动量在看多、看空之间直接翻转；均值回归仍经中性仓过渡（跌破下轨做多、涨破上轨做空，回到 SMA 平仓）。完整类与运行脚本见下文。


In [1]:
import os
proxy = 'http://127.0.0.1:7897'
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy


## 1. 完整类：`BacktestLongShort`

下面是书中用于多空事件驱动回测的完整类（SMA / 动量 / 均值回归）。基类从同目录的 `BacktestBase.py` 导入。


In [ ]:
#
# Python Script with Long-Short Class
# for Event-Based Backtesting
#
# Python for Algorithmic Trading
# (c) Dr. Yves J. Hilpisch
# The Python Quants GmbH
#
from BacktestBase import BacktestBase


class BacktestLongShort(BacktestBase):
    '''多空事件驱动回测。

    仓位取值：1 多头、0 中性、-1 空头。
    go_long / go_short：若已持反向仓，先平仓再按 units 或 amount 开仓；
    amount='all' 表示用当前全部现金。不做分批加仓。
    SMA / 动量在看多与看空之间直接翻转；均值回归可经中性仓过渡。
    类不检查资金是否充足（简化假设）。
    '''

    def go_long(self, bar, units=None, amount=None):
        '''开多：若当前为空头则先平空，再买入。'''
        if self.position == -1:
            self.place_buy_order(bar, units=-self.units)
        if units:
            self.place_buy_order(bar, units=units)
        elif amount:
            if amount == 'all':
                amount = self.amount
            self.place_buy_order(bar, amount=amount)

    def go_short(self, bar, units=None, amount=None):
        '''开空：若当前为多头则先平多，再卖出。'''
        if self.position == 1:
            self.place_sell_order(bar, units=self.units)
        if units:
            self.place_sell_order(bar, units=units)
        elif amount:
            if amount == 'all':
                amount = self.amount
            self.place_sell_order(bar, amount=amount)

    def run_sma_strategy(self, SMA1, SMA2):
        '''回测双均线（SMA）多空策略。

        参数
        ====
        SMA1, SMA2: int
            短均线、长均线窗口（交易日）
        '''
        msg = f'\n\nRunning 双均线（SMA）策略 | SMA1={SMA1} & SMA2={SMA2}'
        msg += f'\nfixed costs {self.ftc} | '
        msg += f'proportional costs {self.ptc}'
        print(msg)
        print('=' * 55)
        self.position = 0  # 初始中性
        self.trades = 0  # 尚未成交
        self.amount = self.initial_amount  # 重置为初始资金
        self.data['SMA1'] = self.data['price'].rolling(SMA1).mean()
        self.data['SMA2'] = self.data['price'].rolling(SMA2).mean()

        for bar in range(SMA2, len(self.data)):
            if self.position in [0, -1]:
                if self.data['SMA1'].iloc[bar] > self.data['SMA2'].iloc[bar]:
                    self.go_long(bar, amount='all')
                    self.position = 1  # 转入多头
            if self.position in [0, 1]:
                if self.data['SMA1'].iloc[bar] < self.data['SMA2'].iloc[bar]:
                    self.go_short(bar, amount='all')
                    self.position = -1  # 转入空头
        self.close_out(bar)

    def run_momentum_strategy(self, momentum):
        '''回测动量多空策略。

        参数
        ====
        momentum: int
            计算均值收益所用的交易日窗口
        '''
        msg = f'\n\nRunning 动量策略 | {momentum} days'
        msg += f'\nfixed costs {self.ftc} | '
        msg += f'proportional costs {self.ptc}'
        print(msg)
        print('=' * 55)
        self.position = 0  # 初始中性
        self.trades = 0  # 尚未成交
        self.amount = self.initial_amount  # 重置为初始资金
        self.data['momentum'] = self.data['return'].rolling(momentum).mean()
        for bar in range(momentum, len(self.data)):
            if self.position in [0, -1]:
                if self.data['momentum'].iloc[bar] > 0:
                    self.go_long(bar, amount='all')
                    self.position = 1  # 转入多头
            if self.position in [0, 1]:
                if self.data['momentum'].iloc[bar] <= 0:
                    self.go_short(bar, amount='all')
                    self.position = -1  # 转入空头
        self.close_out(bar)

    def run_mean_reversion_strategy(self, SMA, threshold):
        '''回测均值回归多空策略。

        参数
        ====
        SMA: int
            均线窗口（交易日）
        threshold: float
            相对 SMA 的绝对偏离阈值（价格单位）
        '''
        msg = f'\n\nRunning 均线窗口策略 | '
        msg += f'SMA={SMA} & thr={threshold}'
        msg += f'\nfixed costs {self.ftc} | '
        msg += f'proportional costs {self.ptc}'
        print(msg)
        print('=' * 55)
        self.position = 0  # 初始中性
        self.trades = 0  # 尚未成交
        self.amount = self.initial_amount  # 重置为初始资金
        self.data['SMA'] = self.data['price'].rolling(SMA).mean()

        for bar in range(SMA, len(self.data)):
            if self.position == 0:
                if (self.data['price'].iloc[bar] < self.data['SMA'].iloc[bar] - threshold):
                    self.go_long(bar, amount=self.initial_amount)
                    self.position = 1
                elif (self.data['price'].iloc[bar] > self.data['SMA'].iloc[bar] + threshold):
                    self.go_short(bar, amount=self.initial_amount)
                    self.position = -1
            elif self.position == 1:
                if self.data['price'].iloc[bar] >= self.data['SMA'].iloc[bar]: # 已经持有多头时，价格一回到均线就平掉
                    self.place_sell_order(bar, units=self.units)
                    self.position = 0
            elif self.position == -1:
                if self.data['price'].iloc[bar] <= self.data['SMA'].iloc[bar]: # 已经持有空头时，价格一回到均线就平掉
                    self.place_buy_order(bar, units=-self.units)
                    self.position = 0
        self.close_out(bar)


## 2. 运行

先在无交易成本下跑三个策略，再在固定成本 10 USD、比例成本 1% 下重跑。标的与纯多头小节一致（`AAPL.O`，2010–2019），便于对照。需要能访问 `hilpisch.com` 的样例 CSV。


In [3]:
def run_strategies(lsbt):
    lsbt.run_sma_strategy(SMA1=42, SMA2=252)
    lsbt.run_momentum_strategy(momentum=60)
    lsbt.run_mean_reversion_strategy(SMA=50, threshold=5)


lsbt = BacktestLongShort(
    symbol='AAPL.O',
    start='2010-1-1',
    end='2019-12-31',
    amount=10000,
    verbose=False,
)
run_strategies(lsbt)




Running 双均线（SMA）策略 | SMA1=42 & SMA2=252
fixed costs 0.0 | proportional costs 0.0
Final balance   [$] 45631.83
Net Performance [%] 356.32
Trades Executed [#] 18.00


Running 动量策略 | 60 days
fixed costs 0.0 | proportional costs 0.0
Final balance   [$] 69059.07
Net Performance [%] 590.59
Trades Executed [#] 242.00


Running 均线窗口策略 | SMA=50 & thr=5
fixed costs 0.0 | proportional costs 0.0
Final balance   [$] 5216.60
Net Performance [%] -47.83
Trades Executed [#] 102.00


In [4]:
# 交易成本：固定 10 USD + 比例 1%
lsbt = BacktestLongShort(
    symbol='AAPL.O',
    start='2010-1-1',
    end='2019-12-31',
    amount=10000,
    ftc=10.0,
    ptc=0.01,
    verbose=False,
)
run_strategies(lsbt)




Running 双均线（SMA）策略 | SMA1=42 & SMA2=252
fixed costs 10.0 | proportional costs 0.01
Final balance   [$] 38369.65
Net Performance [%] 283.70
Trades Executed [#] 18.00


Running 动量策略 | 60 days
fixed costs 10.0 | proportional costs 0.01
Final balance   [$] 4153.95
Net Performance [%] -58.46
Trades Executed [#] 242.00


Running 均线窗口策略 | SMA=50 & thr=5
fixed costs 10.0 | proportional costs 0.01
Final balance   [$] -5893.69
Net Performance [%] -158.94
Trades Executed [#] 102.00


上面脚本会在输出区打印三组策略结果。通过对比无交易成本和有交易成本的回测，可以得出：
- 交易成本会侵蚀业绩；交易越频，拖累越大。
- 允许做空并不保证优于纯多头：本例中多数配置在扣除成本后弱于上一节的 long-only 结果。
- 有成本时均值回归甚至可把余额打穿至负债；实盘需设亏损与负债上限。


## 3. 结果分析

有无交易成本下，纯多头与多空同参数对照如下（初始资金 10000 USD；有成本为固定 10 USD、比例 1%；标的 `AAPL.O`）。成交笔数不随成本变化。末列“加入成本后净收益减少”定义为 `(无成本净收益 − 有成本净收益) / |无成本净收益|`。允许做空并不自动提高净收益。

| 策略 | 模式 | 成交笔数 | 无成本净收益 | 有成本净收益 | 加入成本后净收益减少 |
| --- | --- | ---: | ---: | ---: | ---: |
| 双均线（SMA）42/252 | 纯多头 | 10 | 462.05% | 419.60% | 9.19% |
| 双均线（SMA）42/252 | 多空 | 18 | 356.32% | 283.70% | 20.38% |
| 动量 60 日 | 纯多头 | 122 | 729.69% | 129.07% | 82.31% |
| 动量 60 日 | 多空 | 242 | 590.59% | -58.46% | 109.90% |
| 均线窗口 50/5 | 纯多头 | 41 | 15.44% | -25.99% | 268.33% |
| 均线窗口 50/5 | 多空 | 102 | -47.83% | -158.94% | 232.30% |


1. 总览

    同参数，多空的无成本净收益全面低于纯多头，有成本后差距更大。在 2010–2019 的 AAPL 上，放开做空没有提高这三套规则的样本内业绩。

    成交笔数上，多空约为纯多头的 1.8–2.5 倍。原因是 SMA / 动量在看空时不再持现金，而是先平仓再开反向仓，一次信号翻转常计两笔；均线窗口则额外增加“涨破上轨做空、回到均线平空”的路径。笔数上升会放大固定成本与比例成本。

2. 按策略

    双均线：交叉稀少，两种模式笔数都最低。纯多头成本拖累仅约 9%，多空约 20%，是三组里最耐交易成本的。'多空' 相比 '做多' 少，主要因为上行市中的空头段在消耗多头利润, 低频趋势规则更吃得起摩擦。

    动量：无成本时两种模式都最高收益都最高，也最怕成本。纯多头拖累约 82%，多空约 110%。60 日收益均值频繁过零，换手高；多空再把每次翻转变成平仓加开反向，导致交易成本翻倍。高换手时序动量的账面超额，经不起真实交易成本的考验。

    均线窗口：'做多' 无成本时有约 15% 的净收益，有成本时亏损；'多空' 无成本已经亏损，有成本亏得更多。根本原因在于 '逆势规则撞上趋势市场'。多空补上了“涨过头做空”，但在持续上行里空头段往往更亏钱；再叠加近百笔交易成本，交易产生的收益被消耗殆尽。

3. 对照

    - 模式：同信号下，多空提高的是 '方向自由度' 与 '成交次数'，不是自动提高夏普或净收益；趋势市里空头暴露常常是减分项。
    - 频率：笔数与成本拖累同向。SMA 耐交易成本，动量不耐交易成本；均线窗口在错误市场状态（趋势市场）下，低频也救不了。
    - 评价：叠加交易成本后, SMA 纯多头 > 动量多空 = 均线窗口 （后两者均失效）。比较策略必须同时考虑交易成本。


## 4. 总结

上一节强调业绩有三面：
- 方向命中率、
- 市场择时、
- 策略触发的交易次数。

多空框架在此之上又多一维：
- 反向仓是否改善净收益。

本例说明，灵活性本身不是优势；成本、标的趋势与平仓规则共同决定结果。
